## Tip: 
### Use Kaggle Notebook>Settings>Accelerator>GPU T4 x2 for free with limitations to train the YOLO26-seg Model

---

In [1]:
# ============================================================
# CELL 1 — INSTALL REQUIRED LIBRARIES
# ============================================================

%pip install -q "ultralytics==8.4.166" \
    opencv-python \
    pillow \
    requests \
    rapidfuzz \
    matplotlib \
    pandas \
    tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 427.2 kB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 6.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 34.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.0/96.0 kB 5.7 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import os

print("===== /kaggle/input =====")

for root, dirs, files in os.walk("/kaggle/input"):
    level = root.replace("/kaggle/input", "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{root}/")

    for file in files[:20]:
        print(f"{indent}  {file}")

print("\n===== FILE COUNT =====")

all_files = list(
    Path("/kaggle/input").rglob("*")
)

files_only = [
    p for p in all_files
    if p.is_file()
]

print(
    "Total files:",
    len(files_only)
)

print(
    "Total size:",
    sum(
        p.stat().st_size
        for p in files_only
    ) / (1024**2),
    "MB"
)

===== /kaggle/input =====
/kaggle/input/
  /kaggle/input/datasets/
    /kaggle/input/datasets/sameer088/
      /kaggle/input/datasets/sameer088/filteredbigbookspinedetection/
        /kaggle/input/datasets/sameer088/filteredbigbookspinedetection/valid/
          /kaggle/input/datasets/sameer088/filteredbigbookspinedetection/valid/labels/
            book_000914.txt
            book_000502.txt
            book_000732.txt
            book_000909.txt
            book_000438.txt
            book_000557.txt
            book_000839.txt
            book_000370.txt
            book_000643.txt
            book_000597.txt
            book_000043.txt
            book_000534.txt
            book_000366.txt
            book_000997.txt
            book_000481.txt
            book_000355.txt
            book_000664.txt
            book_000788.txt
            book_000227.txt
            book_000605.txt
          /kaggle/input/datasets/sameer088/filteredbigbookspinedetection/valid/images/
            b

In [ ]:
from pathlib import Path

# ============================================================
# DATASET PATH
# ============================================================

DATASET_ROOT = Path(
    "/kaggle/input/datasets/sam..."
)

print("Dataset root:")
print(DATASET_ROOT)

print("\nExists:", DATASET_ROOT.exists())

In [4]:
# ============================================================
# VERIFY DATASET STRUCTURE
# ============================================================

required_paths = [
    DATASET_ROOT / "train" / "images",
    DATASET_ROOT / "train" / "labels",

    DATASET_ROOT / "valid" / "images",
    DATASET_ROOT / "valid" / "labels",

    DATASET_ROOT / "test" / "images",
    DATASET_ROOT / "test" / "labels",
]

for path in required_paths:
    print(
        f"{path} -> {path.exists()}"
    )

/kaggle/input/datasets/sameer088/filteredbigbookspinedetection/train/images -> True
/kaggle/input/datasets/sameer088/filteredbigbookspinedetection/train/labels -> True
/kaggle/input/datasets/sameer088/filteredbigbookspinedetection/valid/images -> True
/kaggle/input/datasets/sameer088/filteredbigbookspinedetection/valid/labels -> True
/kaggle/input/datasets/sameer088/filteredbigbookspinedetection/test/images -> True
/kaggle/input/datasets/sameer088/filteredbigbookspinedetection/test/labels -> True


In [5]:
# ============================================================
# VERIFY COUNTS
# ============================================================

IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".webp",
    ".bmp"
}


def get_images(folder):

    return [
        p
        for p in folder.iterdir()

        if (
            p.is_file()
            and p.suffix.lower()
            in IMAGE_EXTENSIONS
        )
    ]


EXPECTED_COUNTS = {
    "train": 16484,
    "valid": 1069,
    "test": 499,
}


for split, expected in EXPECTED_COUNTS.items():

    images_dir = (
        DATASET_ROOT
        / split
        / "images"
    )

    labels_dir = (
        DATASET_ROOT
        / split
        / "labels"
    )

    images = get_images(images_dir)

    labels = list(
        labels_dir.glob("*.txt")
    )

    print(
        f"{split.upper():5s} | "
        f"Images: {len(images):5d} | "
        f"Labels: {len(labels):5d} | "
        f"Expected: {expected:5d}"
    )

TRAIN | Images: 16313 | Labels: 16313 | Expected: 16484
VALID | Images:  1053 | Labels:  1053 | Expected:  1069
TEST  | Images:   492 | Labels:   492 | Expected:   499


In [6]:
# ============================================================
# CREATE YOLO DATASET YAML
# ============================================================


yaml_path = Path(
    "/kaggle/working/book_spines.yaml"
)

yaml_content = f"""
path: {DATASET_ROOT}

train: train/images
val: valid/images
test: test/images

names:
  0: book_spine
"""

yaml_path.write_text(
    yaml_content.strip(),
    encoding="utf-8"
)

print(yaml_path.read_text())

path: /kaggle/input/datasets/sameer088/filteredbigbookspinedetection

train: train/images
val: valid/images
test: test/images

names:
  0: book_spine


In [7]:
# ============================================================
# IMPORTS + GPU CONFIGURATION
# ============================================================

import os
import sys
import random
import shutil

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import torch

from pathlib import Path
from tqdm.auto import tqdm
from ultralytics import YOLO


print("Python:")
print(sys.version)

print("\nPyTorch:")
print(torch.__version__)

print("\nCUDA available:")
print(torch.cuda.is_available())

print("\nNumber of GPUs:")
print(torch.cuda.device_count())


if torch.cuda.is_available():

    DEVICE = [0, 1]

    for i in range(torch.cuda.device_count()):
        print(
            f"GPU {i}:",
            torch.cuda.get_device_name(i)
        )

else:

    DEVICE = "cpu"

    print("\nWARNING: GPU is not available.")


# ============================================================
# REPRODUCIBILITY
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Python:
3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]

PyTorch:
2.11.0+cu128

CUDA available:
True

Number of GPUs:
2
GPU 0: Tesla T4
GPU 1: Tesla T4


In [ ]:
from pathlib import Path

MODEL_ROOT = Path(
    "/kaggle/input/models/sam..."
)

print("Model files:")
for p in MODEL_ROOT.rglob("*"):
    if p.is_file():
        print(p)

In [ ]:
MODEL_PATH = Path("/kaggle/input/models/sam...pt")

In [17]:
from ultralytics import YOLO


seg_model = YOLO(MODEL_PATH, task="segment")

print("Trained YOLO26n-seg model loaded successfully.")

Trained YOLO26n-seg model loaded successfully.


In [18]:
# ============================================================
# TRAINING OUTPUT DIRECTORIES
# ============================================================

RUNS_DIR = Path(
    "/kaggle/working/runs"
)

CHECKPOINT_DIR = Path(
    "/kaggle/working/checkpoints"
)

RUNS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


RUN_NAME = (
    "book_spine_segmentation_18k"
)

In [19]:
# ============================================================
# CHECKPOINT CALLBACK
# ============================================================

def archive_checkpoints(trainer):

    try:

        epoch = trainer.epoch + 1

        # Only archive every 10 epochs
        if epoch % 10 != 0:
            return


        run_dir = Path(
            trainer.save_dir
        )

        weights_dir = (
            run_dir / "weights"
        )


        # Destination
        archive_dir = (
            CHECKPOINT_DIR
            / f"epoch_{epoch:03d}"
        )

        archive_dir.mkdir(
            parents=True,
            exist_ok=True
        )


        # ----------------------------------------------------
        # BEST.PT
        # ----------------------------------------------------

        best_path = (
            weights_dir / "best.pt"
        )

        if best_path.exists():

            shutil.copy2(
                best_path,
                archive_dir / "best.pt"
            )


        # ----------------------------------------------------
        # LAST.PT
        # ----------------------------------------------------

        last_path = (
            weights_dir / "last.pt"
        )

        if last_path.exists():

            shutil.copy2(
                last_path,
                archive_dir / "last.pt"
            )


        # ----------------------------------------------------
        # PERIODIC EPOCH CHECKPOINT
        # ----------------------------------------------------

        periodic_path = (
            weights_dir
            / f"epoch{epoch}.pt"
        )

        if periodic_path.exists():

            shutil.copy2(
                periodic_path,
                archive_dir
                / f"epoch{epoch}.pt"
            )


        # ----------------------------------------------------
        # RESULTS CSV
        # ----------------------------------------------------

        results_csv = (
            run_dir / "results.csv"
        )

        if results_csv.exists():

            shutil.copy2(
                results_csv,
                archive_dir / "results.csv"
            )


        print(
            "\n"
            + "=" * 70
        )

        print(
            f"CHECKPOINT BACKUP: "
            f"EPOCH {epoch}"
        )

        print(
            archive_dir
        )

        print(
            "=" * 70
        )


    except Exception as e:

        # Never terminate training because
        # checkpoint archiving failed.

        print(
            f"[Checkpoint warning] {e}"
        )

In [20]:
# ============================================================
# REGISTER CALLBACK
# ============================================================

seg_model.add_callback(
    "on_model_save",
    archive_checkpoints
)

print(
    "Checkpoint callback registered."
)

Checkpoint callback registered.


# Save the model at each epochs as

- best.pt
- last.pt
- every 10th epoch

In [21]:
# ============================================================
# TRAIN YOLO26N-SEG
# ============================================================

IMG_SIZE = 960
EPOCHS = 200
BATCH_SIZE = 16
PATIENCE = 25
WORKERS = 0
SAVE_PERIOD = 10


results = seg_model.train(

    # Dataset
    data=str(
        yaml_path
    ),

    # Training
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,

    # Device
    device=DEVICE,

    # Early stopping
    patience=PATIENCE,

    # Data loader
    workers=WORKERS,
    cache=False,

    # Mixed precision
    amp=True,

    # Reproducibility
    seed=SEED,

    # Output
    project=str(
        RUNS_DIR
    ),

    name=RUN_NAME,

    exist_ok=True,

    # Plots
    plots=True,

    # Checkpoints
    save=True,
    save_period=SAVE_PERIOD
)

New https://pypi.org/project/ultralytics/8.4.173 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.166 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
                                                        CUDA:1 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/book_spines.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=200, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=960, iou=0.7, kobj=1.0, line_width=None, lr0=0.

/usr/local/lib/python3.13/dist-packages/ray/train/_internal/session.py:683: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(



      Epoch    GPU_mem   box_loss   seg_loss   cls_loss    l1_loss   sem_loss  Instances       Size
      2/200       8.2G     0.6834     0.9483     0.3477   0.004132     0.4431        128        960: 100% ━━━━━━━━━━━━ 1020/1020 1.2it/s 13:510.7ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 33/33 1.8it/s 18.8s0.7s
                   all       1053      18249      0.976      0.956       0.97      0.846      0.974      0.954       0.97      0.793

      Epoch    GPU_mem   box_loss   seg_loss   cls_loss    l1_loss   sem_loss  Instances       Size
      3/200      9.66G      0.738     0.9954     0.3783   0.004651      0.475        232        960: 100% ━━━━━━━━━━━━ 1020/1020 1.2it/s 14:200.7ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 33/33 1.9it/s 17.7s0.6s
                 

[rank0]: Traceback (most recent call last):
[rank0]:   File "/root/.config/Ultralytics/DDP/_temp_i3yrbxtz140075780209504.py", line 16, in <module>
[rank0]:     trainer.train()
[rank0]:     ~~~~~~~~~~~~~^^
[rank0]:   File "/usr/local/lib/python3.13/dist-packages/ultralytics/engine/trainer.py", line 257, in train
[rank0]:     self._do_train()
[rank0]:     ~~~~~~~~~~~~~~^^
[rank0]:   File "/usr/local/lib/python3.13/dist-packages/ultralytics/engine/trainer.py", line 510, in _do_train
[rank0]:     for i, batch in pbar:
[rank0]:                     ^^^^
[rank0]:   File "/usr/local/lib/python3.13/dist-packages/ultralytics/utils/tqdm.py", line 379, in __iter__
[rank0]:     for item in self.iterable:
[rank0]:                 ^^^^^^^^^^^^^
[rank0]:   File "/usr/local/lib/python3.13/dist-packages/ultralytics/data/build.py", line 86, in __iter__
[rank0]:     yield next(self.iterator)
[rank0]:           ~~~~^^^^^^^^^^^^^^^
[rank0]:   File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/da

CalledProcessError: Command '['/usr/bin/python3', '-m', 'torch.distributed.run', '--nproc_per_node', '2', '--master_port', '27907', '/root/.config/Ultralytics/DDP/_temp_i3yrbxtz140075780209504.py']' returned non-zero exit status 1.